# Option A: Multimodal Embeddings

This notebook implements pure multimodal embeddings. Instead of converting images to text, it passes raw image pixels through a Vision Encoder to project them into the same vector space as the text queries. We use `jinaai/jina-clip-v1` for state-of-the-art text-image alignment.

It evaluates the Recall@5 on the training set and saves the ordered candidate lists to the cache so they can be reused or inspected later.

Current Model Selections:
- [x] `jinaai/jina-clip-v1`
- [ ] `vidore/colpali`: State-of-the-art late-interaction model for retrieving documents, charts, and tables without OCR.
- [ ] `BAAI/bge-visualized-m3`: BAAI's multimodal extension that maps images directly into the BGE-M3 text space.
- [ ] `google/siglip-so400m-patch14-384`: Google's successor to CLIP with higher resolution patches for superior dense feature extraction.

Cached is saved in `../outputs/cache/multimodal_scores`.


In [1]:
# === IMPORT LIBRARIES ===
import sys
import os
import json
import torch
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
from PIL import Image
import torch.nn.functional as F
from transformers import AutoModel

In [2]:
# === SETUP PATHS ===

# Disable VLM recaptioning to ensure we are testing pure raw image embeddings
os.environ["DISABLE_VLM"] = "1"

# Setup paths
base_dir = Path("..").resolve()
sys.path.append(str(base_dir))
from src.data.loader import load_jsonl

data_dir = base_dir / "data"
cache_dir = base_dir / "outputs/cache/multimodal_scores"
cache_dir.mkdir(parents=True, exist_ok=True)

In [3]:
# === CHOOSE MODEL HERE ===
print("🚀 Loading model...")
model_name = 'jinaai/jina-clip-v2'
model_subname = model_name.split('/')[-1]
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Initialize model with trust_remote_code
model = AutoModel.from_pretrained(model_name, trust_remote_code=True).to(device)
model.eval()
print("✅ Model loaded successfully!")

🚀 Loading model...


config.json: 0.00B [00:00, ?B/s]

I0000 00:00:1779414465.169270  469558 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779414465.197470  469558 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779414465.849140  469558 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


model.safetensors:   0%|          | 0.00/1.73G [00:00<?, ?B/s]

/home/emmy/.cache/huggingface/modules/transformers_modules/jinaai/jina-clip-implementation/39e6a55ae971b59bea6e44675d237c99762e7ee2/modeling_clip.py:140: UserWarning: Flash attention is not installed. Check https://github.com/Dao-AILab/flash-attention?tab=readme-ov-file#installation-and-features for installation instructions, disabling
  warnings.warn(
/home/emmy/.cache/huggingface/modules/transformers_modules/jinaai/jina-clip-implementation/39e6a55ae971b59bea6e44675d237c99762e7ee2/modeling_clip.py:175: UserWarning: xFormers is not installed. Check https://github.com/facebookresearch/xformers?tab=readme-ov-file#installing-xformers for installation instructions, disabling
  warnings.warn(
A new version of the following files was downloaded from https://huggingface.co/jinaai/xlm-roberta-flash-implementation:
- stochastic_depth.py
- mlp.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A

✅ Model loaded successfully!


In [ ]:
datasets = {
    "train": data_dir / "train.jsonl",
    "test": data_dir / "test.jsonl"
}

for split_name, path in datasets.items():
    if not path.exists():
        print(f"⚠️ Dataset not found at {path}. Skipping.")
        continue
        
    print(f"\n⚙️ Generating multimodal scores for {split_name} set...")
    data = load_jsonl(str(path))
    
    output_file = cache_dir / f"{model_subname}_multimodal_scores_{split_name}.json"
    cache_dict = {}
    if output_file.exists():
        with open(output_file, "r") as f:
            cache_dict = json.load(f)
            
    hits_at_5 = 0
    total_samples = 0
    
    with torch.no_grad():
        for sample in tqdm(data, desc=f"Computing {split_name}"):
            if str(sample.q_id) in cache_dict:
                ordered_ids = cache_dict[str(sample.q_id)]
            else:
                query_emb = model.encode_text([sample.question])
                query_tensor = torch.tensor(query_emb).to(device)
                
                candidates = []
                candidate_ids = []
                
                if sample.text_quotes:
                    texts = [tq.text for tq in sample.text_quotes]
                    text_ids = [tq.quote_id for tq in sample.text_quotes]
                    text_embs = model.encode_text(texts)
                    candidates.append(torch.tensor(text_embs).to(device))
                    candidate_ids.extend(text_ids)
                
                if sample.img_quotes:
                    img_ids = []
                    images = []
                    for iq in sample.img_quotes:
                        img_path = base_dir / "data/images" / iq.img_path
                        if img_path.exists():
                            try:
                                img = Image.open(img_path).convert("RGB")
                                images.append(img)
                                img_ids.append(iq.quote_id)
                            except Exception as e:
                                pass
                    
                    if images:
                        img_embs = model.encode_image(images)
                        candidates.append(torch.tensor(img_embs).to(device))
                        candidate_ids.extend(img_ids)
                
                if not candidates:
                    continue
                    
                all_candidates_tensor = torch.cat(candidates, dim=0)
                
                query_tensor = F.normalize(query_tensor, p=2, dim=1)
                all_candidates_tensor = F.normalize(all_candidates_tensor, p=2, dim=1)
                
                sims = (query_tensor @ all_candidates_tensor.T).squeeze(0)
                sorted_idx = torch.argsort(sims, descending=True).cpu().numpy()
                ordered_ids = [candidate_ids[i] for i in sorted_idx]
                
                cache_dict[str(sample.q_id)] = ordered_ids
            
            if sample.gold_quotes:
                top_5 = set(ordered_ids[:5])
                gold = set(sample.gold_quotes)
                overlap = len(top_5.intersection(gold))
                hits_at_5 += overlap
                total_samples += len(gold)
                
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(cache_dict, f, indent=2)
    print(f"✅ Cache saved to {output_file}")

    if total_samples > 0:
        recall = hits_at_5 / total_samples
        print(f"\n📊 {model_subname} Validation Recall@5: {recall:.4f}\n")



⚙️ Generating multimodal scores for train set...


Computing train:   0%|          | 0/2055 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/584 [00:00<?, ?B/s]

✅ Cache saved to /data220_2/emmy/mlbio/hw3/outputs/cache/multimodal_scores/jina-clip-v2_multimodal_scores_train.json

📊 jina-clip-v2 Validation Recall@5: 0.3416


⚙️ Generating multimodal scores for test set...


Computing test:   0%|          | 0/1798 [00:00<?, ?it/s]

✅ Cache saved to /data220_2/emmy/mlbio/hw3/outputs/cache/multimodal_scores/jina-clip-v2_multimodal_scores_test.json


: 